# 06 Heat Vulnerability Exploration

            This notebook builds and interprets the Heat Vulnerability Index from exposure, sensitivity,
            and low adaptive capacity indicators using `config/ahp_weights.yml`.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
if (cwd / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd))
elif (cwd / "notebooks" / "notebook_helpers.py").exists():
    sys.path.insert(0, str(cwd / "notebooks"))
else:
    raise FileNotFoundError("Could not find notebook_helpers.py. Run this notebook from the project root or notebooks folder.")

from notebook_helpers import (
    PROJECT_ROOT,
    add_project_root_to_path,
    command_string,
    find_files,
    load_yaml_config,
    notebook_metadata,
    path_status,
    plot_raster,
    print_path_status,
    project_path,
    raster_info,
    raster_stats,
    read_vector,
    run_command,
)

add_project_root_to_path()
RUN_COMMANDS = False  # Change to True only when you want notebook cells to execute CLI scripts.
YEAR = 2023
notebook_metadata("Vulnerability Exploration", YEAR)

## Step 1 - Load HVI weights

In [ ]:
weights = load_yaml_config("config/ahp_weights.yml")
            weights

## Step 2 - Check HVI input layers

In [ ]:
import pandas as pd

            hvi_inputs = {
                "LST mean / exposure": f"data/processed/lst/lst_ibadan_{YEAR}_celsius.tif",
                "UHI intensity / exposure": f"data/processed/uhi/uhi_intensity_{YEAR}.tif",
                "hotspot membership / exposure": f"data/processed/vulnerability/hotspot_membership_{YEAR}.tif",
                "population density / sensitivity": f"data/processed/vulnerability/population_density_{YEAR}.tif",
                "built-up density / sensitivity": f"data/processed/vulnerability/built_up_density_{YEAR}.tif",
                "NDVI / adaptive capacity": f"data/processed/indices/ndvi_{YEAR}.tif",
                "distance to green space / adaptive capacity": f"data/processed/vulnerability/distance_to_green_space_{YEAR}.tif",
                "distance to water / adaptive capacity": f"data/processed/vulnerability/distance_to_water_{YEAR}.tif",
            }
            pd.DataFrame(path_status(hvi_inputs))

## Step 3 - Preview optional supporting prep commands

In [ ]:
run_command(["python", "scripts/02b_prepare_worldpop.py", "--year", YEAR], dry_run=True)
            run_command(["python", "scripts/02c_prepare_ghsl.py", "--year", YEAR], dry_run=True)
            run_command(["python", "scripts/02d_prepare_osm_layers.py", "--year", YEAR], dry_run=True)

## Step 4 - Build HVI rasters

In [ ]:
run_command(["python", "scripts/08_build_vulnerability_index.py", "--weights", "config/ahp_weights.yml", "--year", YEAR], dry_run=not RUN_COMMANDS)

## Step 5 - Inspect generated HVI outputs

In [ ]:
hvi_outputs = {
                "Heat Exposure Index": project_path(f"data/processed/vulnerability/heat_exposure_index_{YEAR}.tif"),
                "Sensitivity Index": project_path(f"data/processed/vulnerability/sensitivity_index_{YEAR}.tif"),
                "Adaptive Capacity Index": project_path(f"data/processed/vulnerability/adaptive_capacity_index_{YEAR}.tif"),
                "Heat Vulnerability Index": project_path(f"data/processed/vulnerability/heat_vulnerability_index_{YEAR}.tif"),
            }
            for name, path in hvi_outputs.items():
                print("\n", name)
                if path.exists():
                    print(raster_info(path))
                    print(raster_stats(path))
                    plot_raster(path, f"{name} {YEAR}", cmap="magma_r" if "Vulnerability" in name else "viridis")
                else:
                    print("Not generated yet:", path.relative_to(PROJECT_ROOT))

## Step 6 - Review LGA ranking

In [ ]:
import pandas as pd

            ranking_path = project_path(f"data/processed/tables/ibadan_hvi_ranking_{YEAR}.csv")
            if ranking_path.exists():
                ranking = pd.read_csv(ranking_path)
                display(ranking)
                ranking.plot.barh(x="lga_name", y="hvi_mean", figsize=(8, 6), legend=False, title=f"Ibadan HVI ranking {YEAR}")
            else:
                print("HVI ranking table not found yet.")

## Step 7 - Generate final outputs and report

In [ ]:
run_command(["python", "scripts/09_generate_outputs.py", "--year", YEAR], dry_run=not RUN_COMMANDS)